# Basic Logic and Proof Techniques for Math and ML Theory

This Jupyter notebook is a comprehensive, interactive companion to the [Basic Logic and Proof Techniques](https://theorempath.com/topics/basic-logic-and-proof-techniques) guide from TheoremPath. It covers propositions, implications, quantifiers, proof techniques (direct, contrapositive, contradiction, induction, construction, cases, counterexamples), and includes code to build truth tables, verify equivalences, and solve exercises.

**Why this matters:** Every theorem in mathematics and machine learning theory is built from a small set of proof moves. Understanding these moves makes learning theory, optimization, probability, algorithms, and statistics significantly easier.

The goal is not to memorize proof names, but to look at a statement and ask:
- What are the assumptions?
- What must be shown?
- Which quantifier or implication is the bottleneck?
- Would a direct proof, contrapositive, contradiction, induction, construction, or counterexample make the structure simpler?

## 1. Propositions and Implications

**Proposition:** A statement that is either true or false. For example, "The training error is lower than the validation error by at least 0.2" is a proposition.

**Implication:** $P \Rightarrow Q$ means "if $P$ then $Q$". It is false only when $P$ is true and $Q$ is false.

Let's build a truth table to visualize all cases.

In [1]:
import itertools

def truth_table(variables, expression_func):
    """
    Print a truth table for a logical expression.
    
    Parameters:
    - variables: list of variable names (strings)
    - expression_func: a function that takes a tuple of booleans (in the order of variables) and returns a boolean
    """
    # Generate all combinations of truth values
    rows = list(itertools.product([True, False], repeat=len(variables)))
    
    # Print header
    header = " | ".join(variables) + " | Result"
    print(header)
    print("-" * len(header))
    
    for row in rows:
        result = expression_func(row)
        row_str = " | ".join(str(val) for val in row)
        print(f"{row_str} | {result}")

# Define the implication P -> Q
def implication(values):
    P, Q = values
    return (not P) or Q  # logical equivalence

print("Truth table for P => Q")
truth_table(['P', 'Q'], implication)

Truth table for P => Q
P | Q | Result
--------------
True | True | True
True | False | False
False | True | True
False | False | True


The last two rows (where $P$ is false) are **vacuously true**—the implication is not violated because the premise is false.

### Contrapositive, Converse, and Inverse

- **Contrapositive** of $P \Rightarrow Q$ is $\neg Q \Rightarrow \neg P$. It is logically equivalent to the original.
- **Converse** of $P \Rightarrow Q$ is $Q \Rightarrow P$ (not equivalent in general).
- **Inverse** of $P \Rightarrow Q$ is $\neg P \Rightarrow \neg Q$ (not equivalent in general).

Let's verify the equivalence of an implication and its contrapositive using a truth table.

In [2]:
def contrapositive(values):
    P, Q = values
    return (not Q) <= (not P)  # ¬Q => ¬P

print("Truth table for Contrapositive (¬Q => ¬P)")
truth_table(['P', 'Q'], contrapositive)

# Compare with implication
print("\nAre they equivalent? Let's check all rows:")
rows = list(itertools.product([True, False], repeat=2))
all_equal = all(implication(row) == contrapositive(row) for row in rows)
print(f"Implication and contrapositive are equivalent: {all_equal}")

Truth table for Contrapositive (¬Q => ¬P)
P | Q | Result
--------------
True | True | True
True | False | False
False | True | True
False | False | True

Are they equivalent? Let's check all rows:
Implication and contrapositive are equivalent: True


## 2. Quantifiers

- **Universal quantifier** $\forall x \in S, P(x)$: $P(x)$ holds for every element of $S$.
- **Existential quantifier** $\exists x \in S, P(x)$: at least one element of $S$ satisfies $P(x)$.

**Negating quantifiers** is a high-yield skill:
\[
\neg(\forall x \in S, P(x)) \equiv \exists x \in S \text{ such that } \neg P(x)
\]
\[
\neg(\exists x \in S, P(x)) \equiv \forall x \in S, \neg P(x)
\]

For nested quantifiers, negate from the outside inward.

Let's demonstrate with a finite set of numbers.

In [3]:
# Define a finite universe
S = [1, 2, 3, 4, 5]

# Predicate: even number
def P(x):
    return x % 2 == 0

# Check universal: ∀x in S, P(x)
universal = all(P(x) for x in S)
print(f"∀x in {S}, P(x) (x is even) is {universal}")

# Negation: ∃x in S such that ¬P(x)
neg_universal = any(not P(x) for x in S)
print(f"Negation: ∃x in {S} such that x is odd is {neg_universal}")

# Existential: ∃x in S, P(x)
existential = any(P(x) for x in S)
print(f"∃x in {S}, P(x) (x is even) is {existential}")

# Negation: ∀x in S, ¬P(x)
neg_existential = all(not P(x) for x in S)
print(f"Negation: ∀x in {S}, x is odd is {neg_existential}")

∀x in [1, 2, 3, 4, 5], P(x) (x is even) is False
Negation: ∃x in [1, 2, 3, 4, 5] such that x is odd is True
∃x in [1, 2, 3, 4, 5], P(x) (x is even) is True
Negation: ∀x in [1, 2, 3, 4, 5], x is odd is False


## 3. Choosing a Proof Technique

The following table (from the source) gives a quick guide:

| Statement shape | First proof move to try | Why |
|-----------------|-------------------------|-----|
| $P \Rightarrow Q$ | direct proof | assume $P$, derive $Q$ |
| $P \Rightarrow Q$ where $\neg Q$ is concrete | contrapositive | derive $\neg P$ from $\neg Q$ |
| "No object has property X" | contradiction | assume a witness exists and break something |
| "There exists..." | construction | build the witness |
| "For all natural numbers $n$..." | induction | prove base case and step |
| "For all objects..." and claim seems false | counterexample search | one counterexample disproves it |
| multiple regimes | cases | cover exhaustive alternatives |

We'll now illustrate each technique with examples and code where applicable.

### Direct Proof

**Example:** The sum of two even integers is even.

Let $a = 2m$ and $b = 2n$ for integers $m,n$. Then $a+b = 2(m+n)$, which is even.

Direct proof is best when definitions immediately unpack into useful algebra.

### Contrapositive Proof

**Example:** If $n^2$ is even, then $n$ is even.

Prove the contrapositive: if $n$ is odd, then $n^2$ is odd. Write $n = 2k+1$. Then $n^2 = 4k^2+4k+1 = 2(2k^2+2k)+1$, which is odd. Hence the original implication holds.

Contrapositive is useful when the conclusion has a simple negation, such as "not injective," "not continuous," or "not linearly independent."

### Contradiction

**Example:** $\sqrt{2}$ is irrational.

Assume $\sqrt{2} = a/b$ where $a,b$ are integers with no common factor. Then $2b^2 = a^2$, so $a$ is even. Write $a=2k$. Then $2b^2 = 4k^2 \Rightarrow b^2 = 2k^2$, so $b$ is even. This contradicts the assumption that $a$ and $b$ had no common factor.

Contradiction is powerful, but if a contrapositive proof works cleanly, it is often more informative.

### Induction

**Weak Induction:** To prove $P(n)$ for all $n \ge n_0$, prove the base case $P(n_0)$ and prove $P(k) \Rightarrow P(k+1)$ for every $k \ge n_0$.

**Strong Induction:** Prove enough base cases and prove that $P(n_0), P(n_0+1), \dots, P(k)$ together imply $P(k+1)$.

**Example:** Prove $\sum_{i=1}^{n} i = \frac{n(n+1)}{2}$ for all $n \ge 1$.

Base case ($n=1$): $1 = \frac{1 \cdot 2}{2}$.

Inductive step: Assume $\sum_{i=1}^{k} i = \frac{k(k+1)}{2}$. Then
\[
\sum_{i=1}^{k+1} i = \frac{k(k+1)}{2} + (k+1) = \frac{(k+1)(k+2)}{2}.
\]
Thus the formula holds for all $n$.

Let's verify the formula with a Python loop for a range of values.

In [4]:
def sum_formula(n):
    return n * (n + 1) // 2

def sum_direct(n):
    return sum(range(1, n+1))

# Test for n = 1 to 10
for n in range(1, 11):
    assert sum_formula(n) == sum_direct(n), f"Formula fails for n={n}"
    print(f"n={n}: sum = {sum_direct(n)}, formula = {sum_formula(n)}")
print("All tests passed. The formula holds.")

n=1: sum = 1, formula = 1
n=2: sum = 3, formula = 3
n=3: sum = 6, formula = 6
n=4: sum = 10, formula = 10
n=5: sum = 15, formula = 15
n=6: sum = 21, formula = 21
n=7: sum = 28, formula = 28
n=8: sum = 36, formula = 36
n=9: sum = 45, formula = 45
n=10: sum = 55, formula = 55
All tests passed. The formula holds.


### Construction, Cases, and Counterexamples

- **Constructive Proof:** To prove $\exists x \in S$ with property $P(x)$, explicitly define such an $x$ and verify.
- **Proof by Cases:** Split the situation into exhaustive cases and prove the claim in each case. Cases must cover everything.
- **Counterexample:** To disprove a universal claim $\forall x, P(x)$, find one $x$ such that $\neg P(x)$.

Let's demonstrate a counterexample search. Claim: "If $ab=0$, then $a=0$ and $b=0$." This is false; take $a=0, b=5$.

In [5]:
# Find a counterexample to: ab = 0 => a=0 and b=0
import itertools

# Search over a small range of integers
for a, b in itertools.product(range(-5, 6), repeat=2):
    if a * b == 0 and not (a == 0 and b == 0):
        print(f"Counterexample found: a={a}, b={b}, ab={a*b}")
        break
else:
    print("No counterexample found in the searched range.")

Counterexample found: a=-5, b=0, ab=0


## 4. ML and Math Examples

The source provides a table of proof moves you will see in various areas:

| Area | Proof move you will see |
|------|-------------------------|
| generalization bounds | union bound, concentration, contradiction, quantifier control |
| convex optimization | direct proof from definitions, separating hyperplanes, KKT implications |
| linear algebra | construction of bases, contradiction for independence, induction on dimension |
| probability | law-of-total-probability decompositions, counterexamples to independence claims |
| algorithms | induction on iterations, loop invariants, cases by input branch |
| learning theory | contrapositive and reductions for lower bounds |

## 5. Common Confusions

- **Contrapositive is not the converse.** The contrapositive is equivalent to the original; the converse is not.
- **A proof by examples is not a proof of a universal statement.** You must cover every allowed object.
- **Induction needs a base case.** The step can be true even if no case ever starts.
- **Negating an implication changes it into an and statement.** The negation of $P \Rightarrow Q$ is $P \wedge \neg Q$, not $\neg P \Rightarrow \neg Q$.

## 6. Quick Drills

1. **Negate:** "For every model, there exists a dataset on which it performs well."
   - **Answer:** There exists a model such that for every dataset, it does not perform well.

2. **Identify the proof move:** "Assume the algorithm does not terminate; then construct an infinite strictly decreasing sequence of nonnegative integers."
   - **Answer:** Contradiction, usually with well-ordering or a descent measure.

3. **Find the flaw:** "The theorem holds for $n=1,2,3$, so it holds for all $n$."
   - **Answer:** Examples are not induction. You still need a step proving $P(k) \Rightarrow P(k+1)$.

4. **Contrapositive:** "If a matrix has full column rank, then its columns are linearly independent."
   - **Answer:** If the columns are linearly dependent, then the matrix does not have full column rank.

## 7. Exercises

Try these on your own; solutions are provided in the cells below (run them to check).

### Exercise 1
Write the negation of: "For every $\epsilon > 0$, there exists $\delta > 0$ such that $|x-a| < \delta$ implies $|f(x)-f(a)| < \epsilon$."

### Exercise 2
Prove by induction that $\sum_{i=1}^n i = \frac{n(n+1)}{2}$ for all $n \ge 1$.

### Exercise 3
Disprove: "If $ab=0$, then $a=0$ and $b=0$."

### Exercise 4
Give a proof template for a PAC-style statement of the form: with probability at least $1-\delta$, every hypothesis in a finite class satisfies a deviation bound.

---
**Solutions:**

In [6]:
# Solution Exercise 1: Negation
# Original: ∀ε>0, ∃δ>0, ∀x, (|x-a|<δ ⇒ |f(x)-f(a)|<ε)
# Negation: ∃ε>0, ∀δ>0, ∃x such that |x-a|<δ and |f(x)-f(a)| ≥ ε
print("Exercise 1 Solution:")
print("∃ε>0, ∀δ>0, ∃x such that |x-a|<δ and |f(x)-f(a)| ≥ ε")

Exercise 1 Solution:
∃ε>0, ∀δ>0, ∃x such that |x-a|<δ and |f(x)-f(a)| ≥ ε


In [7]:
# Solution Exercise 2: Induction proof (already given above, but we can re-iterate)
print("Exercise 2 Solution: See the induction section above. The proof is provided.")
# We can also check the formula for many n
for n in range(1, 21):
    assert sum(range(1, n+1)) == n*(n+1)//2
print("Verified for n=1 to 20.")

Exercise 2 Solution: See the induction section above. The proof is provided.
Verified for n=1 to 20.


In [8]:
# Solution Exercise 3: Counterexample
print("Exercise 3 Solution: Counterexample: a=0, b=5 gives ab=0 but b≠0.")
print("Thus the statement 'if ab=0, then a=0 and b=0' is false.")

Exercise 3 Solution: Counterexample: a=0, b=5 gives ab=0 but b≠0.
Thus the statement 'if ab=0, then a=0 and b=0' is false.


In [9]:
# Solution Exercise 4: Template for PAC-style bound
print("Exercise 4 Solution Template:")
print("""
Let H be a finite hypothesis class. For any δ > 0, with probability at least 1-δ over the draw of a training set S of size m,
for all h in H,
    |L_S(h) - L_D(h)| ≤ sqrt( (1/(2m)) * ln(2|H|/δ) )
where L_S is the empirical loss and L_D is the true loss.
The proof uses Hoeffding's inequality and the union bound over all hypotheses.
""")

Exercise 4 Solution Template:

Let H be a finite hypothesis class. For any δ > 0, with probability at least 1-δ over the draw of a training set S of size m,
for all h in H,
    |L_S(h) - L_D(h)| ≤ sqrt( (1/(2m)) * ln(2|H|/δ) )
where L_S is the empirical loss and L_D is the true loss.
The proof uses Hoeffding's inequality and the union bound over all hypotheses.



## 8. References and Next Topics

**Canonical references:**
- Velleman, *How to Prove It*, 3rd ed., Chapters 1-6.
- Hammack, *Book of Proof*, 3rd ed., Chapters 1-10.
- Bloch, *Proofs and Fundamentals*, 2011, Chapters 1-3.

**Further reading:**
- Rosen, *Discrete Mathematics and Its Applications*, logic and proof chapters.
- Sipser, *Introduction to the Theory of Computation*, proof examples in automata and computability.

**Next topics (from the graph):**
- Sets, functions, and relations
- Counting and combinatorics
- SAT, SMT, and automated reasoning

---
**What to remember:**
- An implication fails only when the premise is true and the conclusion is false.
- Contrapositive is equivalent to the original; converse is not.
- Negating quantifiers swaps $\forall$ and $\exists$.
- Induction needs both a base case and a valid step.
- To disprove "for all," find one counterexample.
- Good proof technique follows statement structure, not vibes.